#  Vehicle Classification CNN (Car / Truck / Bus)

**Before running:** Go to `Runtime → Change runtime type → T4 GPU → Save`

Then run each cell **top to bottom** in order.

In [ ]:
# ── CELL 1: Import Libraries ──────────────────────────────────────────
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)
if device.type == 'cpu':
    print('  WARNING: You are on CPU. Training will be very slow.')
    print('   Go to Runtime → Change runtime type → T4 GPU → Save')
else:
    print(' GPU detected — training will be fast!')

In [ ]:
# ── CELL 2: Load Dataset (CIFAR-10, built-in, no upload needed) ───────
#
#  We filter CIFAR-10 to 3 vehicle classes:
#    CIFAR label 1 (automobile) → 0 (car)
#    CIFAR label 9 (truck)      → 1 (truck)
#    CIFAR label 8 (ship)       → 2 (bus)

CIFAR_MAP   = {1: 0, 9: 1, 8: 2}
CLASS_NAMES = ['car', 'truck', 'bus']

train_tf = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])
test_tf = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

class VehicleSubset(torch.utils.data.Dataset):
    def __init__(self, base, class_map):
        self.base      = base
        self.class_map = class_map
        self.indices   = [i for i, (_, l) in enumerate(base) if l in class_map]
    def __len__(self):
        return len(self.indices)
    def __getitem__(self, idx):
        img, lbl = self.base[self.indices[idx]]
        return img, self.class_map[lbl]

print('Downloading CIFAR-10 (only happens once)...')
raw_train = datasets.CIFAR10('./data', train=True,  download=True, transform=train_tf)
raw_test  = datasets.CIFAR10('./data', train=False, download=True, transform=test_tf)

train_ds = VehicleSubset(raw_train, CIFAR_MAP)
test_ds  = VehicleSubset(raw_test,  CIFAR_MAP)

val_n   = int(0.2 * len(train_ds))
train_n = len(train_ds) - val_n
train_ds, val_ds = random_split(train_ds, [train_n, val_n])

# num_workers=0 avoids multiprocessing issues on Colab
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=64, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=64, shuffle=False, num_workers=0)

class_names = CLASS_NAMES
print(f'Train: {len(train_ds):,}  |  Val: {len(val_ds):,}  |  Test: {len(test_ds):,}')
print('Classes:', class_names)

In [ ]:
# ── CELL 3: Preview Sample Images ────────────────────────────────────
imgs, lbls = next(iter(train_loader))

plt.figure(figsize=(14, 2.5))
for i in range(10):
    plt.subplot(1, 10, i + 1)
    img = imgs[i] * 0.5 + 0.5      # un-normalize
    plt.imshow(np.transpose(img.numpy(), (1, 2, 0)))
    plt.title(class_names[lbls[i].item()], fontsize=8)
    plt.axis('off')
plt.suptitle('Sample Training Images', y=1.05)
plt.tight_layout()
plt.show()

In [ ]:
# ── CELL 4: Define CNN Model ──────────────────────────────────────────
#
#  Input:  3 × 128 × 128
#  Conv1:  32 filters → BatchNorm → ReLU → Pool → 32 × 64 × 64
#  Conv2:  64 filters → BatchNorm → ReLU → Pool → 64 × 32 × 32
#  Flatten: 64 × 32 × 32 = 65536
#  FC1:    256 neurons + Dropout(0.5)
#  FC2:    3 neurons  → car / truck / bus

class VehicleCNN(nn.Module):
    def __init__(self):
        super(VehicleCNN, self).__init__()
        self.conv1   = nn.Conv2d(3, 32, 3, padding=1)
        self.bn1     = nn.BatchNorm2d(32)
        self.conv2   = nn.Conv2d(32, 64, 3, padding=1)
        self.bn2     = nn.BatchNorm2d(64)
        self.pool    = nn.MaxPool2d(2, 2)
        # 128×128 → pool → 64×64 → pool → 32×32
        # flat size = 64 * 32 * 32 = 65536
        self.fc1     = nn.Linear(64 * 32 * 32, 256)
        self.dropout = nn.Dropout(0.5)
        self.fc2     = nn.Linear(256, 3)

    def forward(self, x):
        x = self.pool(torch.relu(self.bn1(self.conv1(x))))
        x = self.pool(torch.relu(self.bn2(self.conv2(x))))
        x = x.view(x.size(0), -1)
        x = self.dropout(torch.relu(self.fc1(x)))
        return self.fc2(x)

model = VehicleCNN().to(device)
print(model)
print(f'\nTrainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

In [ ]:
# ── CELL 5: Training Setup ────────────────────────────────────────────
NUM_EPOCHS = 10

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

print(f'Ready to train for {NUM_EPOCHS} epochs.')
print(f'Training samples: {len(train_ds):,}')

In [ ]:
# ── CELL 6: Train the Model ───────────────────────────────────────────
# This will take ~3-5 min on GPU, ~60+ min on CPU

train_losses, val_losses = [], []
train_accs,   val_accs   = [], []

for epoch in range(NUM_EPOCHS):

    # --- Training phase ---
    model.train()
    run_loss, correct, total = 0.0, 0, 0
    for imgs, lbls in train_loader:
        imgs, lbls = imgs.to(device), lbls.to(device)
        optimizer.zero_grad()
        out  = model(imgs)
        loss = criterion(out, lbls)
        loss.backward()
        optimizer.step()
        run_loss += loss.item() * imgs.size(0)
        correct  += (out.argmax(1) == lbls).sum().item()
        total    += lbls.size(0)
    t_loss = run_loss / total
    t_acc  = correct  / total

    # --- Validation phase ---
    model.eval()
    v_loss, v_correct, v_total = 0.0, 0, 0
    with torch.no_grad():
        for imgs, lbls in val_loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            out  = model(imgs)
            loss = criterion(out, lbls)
            v_loss    += loss.item() * imgs.size(0)
            v_correct += (out.argmax(1) == lbls).sum().item()
            v_total   += lbls.size(0)
    vl = v_loss / v_total
    va = v_correct / v_total

    train_losses.append(t_loss);  val_losses.append(vl)
    train_accs.append(t_acc);     val_accs.append(va)
    scheduler.step()

    print(f'Epoch [{epoch+1:2d}/{NUM_EPOCHS}]  '
          f'Train Loss: {t_loss:.4f}  Train Acc: {t_acc*100:.1f}%  '
          f'Val Loss: {vl:.4f}  Val Acc: {va*100:.1f}%')

print('\n Training complete!')

In [ ]:
# ── CELL 7: Plot Loss & Accuracy Curves ──────────────────────────────
epochs_x = range(1, NUM_EPOCHS + 1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(epochs_x, train_losses, 'b-o', label='Train')
ax1.plot(epochs_x, val_losses,   'r-o', label='Validation')
ax1.set_title('Loss Curve')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.legend()
ax1.grid(True)

ax2.plot(epochs_x, [a * 100 for a in train_accs], 'b-o', label='Train')
ax2.plot(epochs_x, [a * 100 for a in val_accs],   'r-o', label='Validation')
ax2.set_title('Accuracy Curve')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.legend()
ax2.grid(True)

plt.suptitle('Training Progress', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ── CELL 8: Evaluate on Test Set ─────────────────────────────────────
model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for imgs, lbls in test_loader:
        imgs, lbls = imgs.to(device), lbls.to(device)
        out = model(imgs)
        all_preds.extend(out.argmax(1).cpu().numpy())
        all_labels.extend(lbls.cpu().numpy())

# Classification Report
print('Classification Report')
print('=' * 50)
print(classification_report(all_labels, all_preds,
                             target_names=class_names, zero_division=0))

# Confusion Matrix
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.tight_layout()
plt.show()

In [ ]:
# ── CELL 9: Save Model ────────────────────────────────────────────────
torch.save(model.state_dict(), 'vehicle_cnn.pth')
print(' Model saved as vehicle_cnn.pth')

In [ ]:
# ── CELL 10: Upload YOUR Image & Predict ─────────────────────────────
# Click 'Choose Files', pick any car / truck / bus photo from your computer

from google.colab import files
from PIL import Image
import io

uploaded = files.upload()      # opens file picker

predict_tf = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

for fname, data in uploaded.items():
    img = Image.open(io.BytesIO(data)).convert('RGB')

    img_tensor = predict_tf(img).unsqueeze(0).to(device)

    model.eval()
    with torch.no_grad():
        out   = model(img_tensor)
        probs = torch.softmax(out, dim=1)[0].cpu().numpy()
        pred  = probs.argmax()

    # Show image with prediction title
    plt.figure(figsize=(4, 4))
    plt.imshow(img)
    plt.title(f'Predicted: {class_names[pred]}  ({probs[pred]*100:.1f}%)',
              fontsize=12, fontweight='bold', color='green')
    plt.axis('off')
    plt.tight_layout()
    plt.show()

    # Show probability bars
    print(f'\nFile: {fname}')
    print('Probabilities:')
    for i, name in enumerate(class_names):
        bar    = '█' * int(probs[i] * 30)
        marker = ' ← predicted' if i == pred else ''
        print(f'  {name:8s}: {probs[i]*100:5.1f}%  {bar}{marker}')